# Notebook 07m (Granite) · merge the two half checkpoints (CPU, a few minutes)

Run this **after Notebook 07a and 07b have both finished** (each shows `HALF ... COMPLETE`). It needs no GPU and no Internet.

**Attach** (both are checkpoint datasets you published from the halves):
- `granite-stage-07a-ckpt` (contains `stage07a_checkpoint_0850_of_0850.zip`)
- `granite-stage-07b-ckpt` (contains `stage07b_checkpoint_0850_of_0850.zip`)

(Shortcut: instead of the two datasets you may attach the Output of the 07a and 07b notebooks directly with Add Input → Notebook Output; the cell searches `/kaggle/input` recursively.)

**Run All.** The cell checks each half (ZIP integrity, one shared run fingerprint, every raw output matches its stored SHA-256, no task in both halves, 850 terminal tasks per half) and writes ONE combined checkpoint ZIP in the exact layout Notebook 07 reads:

`stage07_main_checkpoint_1700_of_1700.zip` (Output) plus `merge_report.json`.

**Keep:** download `stage07_main_checkpoint_1700_of_1700.zip` and publish it as dataset `granite-stage-07-ckpt`. The final, unchanged Notebook 07 attaches it, finds all 1700 tasks done, does no generation, and builds `stage_07_main_dataset`.

If a half is unfinished, the cell stops with a clear message. Set `REQUIRE_COMPLETE = False` only if you deliberately want to merge a partial half; the final Notebook 07 then generates just the missing tasks.


In [ ]:
from __future__ import annotations

import hashlib
import json
import tempfile
from pathlib import Path
from zipfile import BadZipFile, ZIP_DEFLATED, ZipFile

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIRECTORY = Path("/kaggle/working/stage07_main_checkpoints")
EXPECTED_TASKS_PER_HALF = 850
TOTAL_TASKS = 2 * EXPECTED_TASKS_PER_HALF
REQUIRE_COMPLETE = True  # keep True unless you knowingly merge a partial half

CONFIG_MEMBER = "config/generation_frozen.json"
TASK_PREFIX = "checkpoints/tasks/"
RAW_PREFIX = "runs/main/raw/"
TERMINAL_STATES = {"GENERATED", "RAW_RECOVERED", "NO_OUTPUT_RUNTIME_FAILURE"}
HALF_PATTERNS = {"a": "stage07a_checkpoint_*.zip", "b": "stage07b_checkpoint_*.zip"}


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_half(path):
    """Return (prefix, config_bytes, config, records, raws, terminal_count); raise on any inconsistency."""
    with ZipFile(path) as archive:
        bad = archive.testzip()
        if bad is not None:
            raise RuntimeError(f"ZIP integrity failure at {bad}")
        names = archive.namelist()
        if len(set(names)) != len(names):
            raise RuntimeError("duplicate ZIP member")
        configs = [n for n in names if n == CONFIG_MEMBER or n.endswith("/" + CONFIG_MEMBER)]
        if len(configs) != 1:
            raise RuntimeError("expected exactly one frozen generation config")
        prefix = configs[0][:-len(CONFIG_MEMBER)]
        config_bytes = archive.read(configs[0])
        config = json.loads(config_bytes)
        if not isinstance(config.get("run_fingerprint"), str):
            raise RuntimeError("config has no run fingerprint")
        records, raws = {}, {}
        for name in names:
            if not name.startswith(prefix):
                continue
            relative = name[len(prefix):]
            if relative.startswith(TASK_PREFIX) and relative.endswith(".json"):
                records[relative] = archive.read(name)
            elif relative.startswith(RAW_PREFIX) and relative.endswith(".txt"):
                raws[relative] = archive.read(name)
    terminal = 0
    for relative, data in records.items():
        record = json.loads(data)
        task_id = Path(relative).stem
        if record.get("task_id") != task_id:
            raise RuntimeError(f"{relative}: task id does not match file name")
        if record.get("fingerprint") != config["run_fingerprint"]:
            raise RuntimeError(f"{task_id}: record belongs to a different generation fingerprint")
        raw_name = RAW_PREFIX + task_id + ".txt"
        if record.get("expected_output_path") != raw_name:
            raise RuntimeError(f"{task_id}: unexpected output path {record.get('expected_output_path')!r}")
        state = record.get("state")
        if state in ("GENERATED", "RAW_RECOVERED"):
            if raw_name not in raws or sha256_bytes(raws[raw_name]) != record.get("raw_sha256"):
                raise RuntimeError(f"{task_id}: raw output missing or changed")
            terminal += 1
        elif state == "NO_OUTPUT_RUNTIME_FAILURE":
            if raw_name in raws:
                raise RuntimeError(f"{task_id}: unexpected raw output for a no-output task")
            terminal += 1
        elif state != "STARTED":
            raise RuntimeError(f"{task_id}: unknown state {state!r}")
    orphans = {n for n in raws if TASK_PREFIX + Path(n).stem + ".json" not in records}
    if orphans and REQUIRE_COMPLETE:
        raise RuntimeError(f"{len(orphans)} raw outputs without a task record")
    return prefix, config_bytes, config, records, raws, terminal


chosen = {}
for half, pattern in HALF_PATTERNS.items():
    candidates = sorted(INPUT_ROOT.rglob(pattern))
    if not candidates:
        raise RuntimeError(f"Half {half.upper()}: no {pattern} found under {INPUT_ROOT}. "
                           f"Attach dataset granite-stage-07{half}-ckpt.")
    best = None
    for path in candidates:
        try:
            result = read_half(path)
        except (RuntimeError, OSError, ValueError, BadZipFile) as error:
            print(f"Half {half.upper()}: ignored {path}: {error}")
            continue
        print(f"Half {half.upper()}: {path.name} -> {result[5]} terminal tasks, {len(result[3])} records")
        if best is None or result[5] > best[1][5]:
            best = (path, result)
    if best is None:
        raise RuntimeError(f"Half {half.upper()}: no usable checkpoint ZIP")
    chosen[half] = best
    if REQUIRE_COMPLETE and best[1][5] != EXPECTED_TASKS_PER_HALF:
        raise RuntimeError(f"Half {half.upper()} has {best[1][5]}/{EXPECTED_TASKS_PER_HALF} tasks finished. "
                           "Finish that half first (resume Notebook 07" + half + " with its latest checkpoint).")

(path_a, (prefix_a, config_bytes_a, config_a, records_a, raws_a, terminal_a)) = chosen["a"]
(path_b, (prefix_b, config_bytes_b, config_b, records_b, raws_b, terminal_b)) = chosen["b"]
if config_a != config_b:
    differing = sorted(k for k in set(config_a) | set(config_b) if config_a.get(k) != config_b.get(k))
    raise RuntimeError("The two halves were generated under different configurations (fields: "
                       + ", ".join(differing) + "). They must not be merged.")
overlap = {Path(n).stem for n in records_a} & {Path(n).stem for n in records_b}
if overlap:
    raise RuntimeError(f"{len(overlap)} tasks appear in both halves, for example {sorted(overlap)[:3]}")
if len(records_a) > EXPECTED_TASKS_PER_HALF or len(records_b) > EXPECTED_TASKS_PER_HALF:
    raise RuntimeError("A half contains more tasks than its share")

merged_records = {**records_a, **records_b}
merged_raws = {**raws_a, **raws_b}
terminal_total = terminal_a + terminal_b
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
output = OUTPUT_DIRECTORY / f"stage07_main_checkpoint_{terminal_total:04d}_of_{TOTAL_TASKS:04d}.zip"
with tempfile.NamedTemporaryFile(dir=OUTPUT_DIRECTORY, suffix=".tmp", delete=False) as stream:
    temporary = Path(stream.name)
try:
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=3) as archive:
        archive.writestr(prefix_a + CONFIG_MEMBER, config_bytes_a)
        for relative in sorted(merged_records):
            archive.writestr(prefix_a + relative, merged_records[relative])
        for relative in sorted(merged_raws):
            archive.writestr(prefix_a + relative, merged_raws[relative])
    with ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("merged ZIP failed its integrity test")
        names = archive.namelist()
        if len(set(names)) != len(names):
            raise RuntimeError("merged ZIP has duplicate members")
    temporary.replace(output)
finally:
    temporary.unlink(missing_ok=True)

REPORT = {
    "status": "PASS", "run_fingerprint": config_a["run_fingerprint"],
    "half_a": {"file": path_a.name, "sha256": sha256_file(path_a), "terminal_tasks": terminal_a},
    "half_b": {"file": path_b.name, "sha256": sha256_file(path_b), "terminal_tasks": terminal_b},
    "merged": {"file": output.name, "sha256": sha256_file(output), "terminal_tasks": terminal_total,
               "task_records": len(merged_records), "raw_outputs": len(merged_raws)},
    "complete": terminal_total == TOTAL_TASKS,
}
(OUTPUT_DIRECTORY.parent / "merge_report.json").write_text(json.dumps(REPORT, indent=2) + "\n", encoding="utf-8")
print(json.dumps(REPORT, indent=2))
print()
print("MERGE:", "COMPLETE" if REPORT["complete"] else "PARTIAL", f"({terminal_total}/{TOTAL_TASKS} tasks)")
print("Download", output.name, "from Output and publish it as dataset granite-stage-07-ckpt.")
